In [1]:
import pandas as pd
import numpy as np

In [674]:
financial_data = pd.read_csv('financial_data.csv')
prolong = pd.read_csv('prolongations.csv')

**Нам даны 2 датасета, из которых нужно узнать информацию как качественно менеджеры выполняют свою из основных обязонностей – пролонгацией договоров с клиентами (продление договоров с клиентами).**


**prolongations.csv**:

  `id` – id проекта
  
  `month` – последний месяц реализации проекта
  
  `AM` – ФИО ответственного аккаунт-менеджера (данные первичны по отношению к financial_data)

**financial_data.csv**:

  `id` – id проекта
  
  `Причина дубля` – причина, почему строки с одним и тем же id встречаются несколько раз
  
  `Колонки с названием месяца` – сумма отгрузки проекта в данный месяц
  
  `Account` – ФИО ответственного аккаунт-менеджера


In [676]:
# Чтобы работать с этими данными, нужно сделать несколько преобразований

financial_data.columns = [i.lower() for i in financial_data.columns] # чтобы названия месяцев двух датасетов сходились
columns_month = financial_data.columns.to_list()[2:-1]

# Для работы с колонками, как с числами, придётся убрать всё лишние
def clean_data(value):
    
    if pd.isna(value):
        return np.nan
    if isinstance(value, str):
        if value == 'стоп' or value == 'end':
            return np.nan
        elif value == 'в ноль':
            return 0
        else:
            value = float(value.replace('\xa0', '').replace(',', '.'))
            return value

for column in columns_month:
    financial_data[column] = financial_data[column].apply(clean_data).astype('float')

    financial_grouped = financial_data.groupby(['id'], as_index = False).agg({
    **{month: 'sum' for month in columns_month },
    'account': 'first'
})

# В датасете financial_data.csv есть пару условностей, которые нужно учесть.
# Для проектов с "в ноль" берем отгрузку предыдущего месяца (если все части оплаты равны 0)

for i in range(1, len(columns_month )):
    current_month = columns_month [i]
    prev_month = columns_month [i-1]

    # Находим проекты, у которых в текущем месяце 0 и нет отгрузки в следующих месяцах
    if_none_next = (financial_grouped[current_month] == 0) & \
           (financial_grouped[columns_month [i+1:]].isna().all(axis=1))

    financial_grouped.loc[if_none_next, current_month] = financial_grouped.loc[if_none_next, prev_month]


In [678]:
prolongation_year = []

months_2023 = [col for col in columns_month if '2023' in col]

for current_month in months_2023:
    
    # Находим предыдущий месяц
    month_idx = columns_month.index(current_month)
    prev_month = columns_month[month_idx - 1]

    # ----------------------------------------------------------------------------
    # Подсчитаем первый коэффицент пролонгаций для всей компании
    
     # Находим проекты, завершившиеся в предыдущем месяце
    ended_projects = prolong[prolong['month'] == prev_month]['id'].unique()

    # Сумма отгрузок последнего месяца завершившегося проекта
    last_month_sum_k1 = financial_grouped[financial_grouped['id'].isin(ended_projects)][prev_month].sum()

    # Сумма отгрузок тех-же проектов, но на следующий месяц
    prolong_sum_k1 = financial_grouped[financial_grouped['id'].isin(ended_projects)][current_month].sum()

    # Коэффициент пролонгаций 1 в текущем месяце
    prolongation_coefficient_k1 = prolong_sum_k1 / last_month_sum_k1 

# ----------------------------------------------------------------------------

    # Подсчитаем второй коэффицент пролонгаций для всей компании
    
    # Находим проекты, завершившиеся два месяца назад
    if month_idx >= 2:
        two_month_ago = columns_month[month_idx - 2]
        
        ended_projects_two_months_ago = prolong[prolong['month'] == two_month_ago]['id'].unique()

        # Находим проекты, которые на следующий месяц не имели отгрузку
        
        not_prolonged_first_month = financial_grouped[financial_grouped['id'].isin(ended_projects_two_months_ago) & \
                                (financial_grouped[prev_month] == 0)]['id'].unique()
    
        # Сумма отгрузки в месяц завершения
        two_month_ago_sum_k2 = financial_grouped[financial_grouped['id'].isin(not_prolonged_first_month )][two_month_ago].sum()
    
        # Сумма отгрузки в текущйи месяц
        prolonged_sum_two_k2 = financial_grouped[financial_grouped['id'].isin(not_prolonged_first_month)][current_month].sum()

        prolongation_coefficient_k2 = prolonged_sum_two_k2 / two_month_ago_sum_k2

        prolongation_year.append({
            'Месяц': current_month,
            'Коэффициент пролонгаций 1': prolongation_coefficient_k1,
            'Коэффициент пролонгаций 2': prolongation_coefficient_k2})
        
prolongation_year_df = pd.DataFrame(prolongation_year)
    

In [694]:
prolongation_year_manager = []

# Получаем список всех менеджеров
managers = prolong['AM'].unique()

for manager in managers:
    manager_prolong = prolong[prolong['AM'] == manager]
    finance_manager = financial_grouped[financial_grouped['account'] == manager]

    manager_financial_grouped = finance_manager.groupby('id')[columns_month].sum().reset_index()

    for current_month in months_2023:
        
        month_idx = columns_month.index(current_month)
        prev_month = columns_month[month_idx - 1]

# ---------------------------------------------------------------------------------        
        
        # Подсчитаем первый коэффицент пролонгаций по сотрудникам
        
         # Проекты менеджера, завершившиеся в предыдущем месяце
        ended_projects = manager_prolong[manager_prolong['month'] == prev_month]['id'].unique()
        
        # Сумма отгрузки завершившихся проектов в последний месяц
        last_month_sum_k1 = manager_financial_grouped[manager_financial_grouped['id'].isin(ended_projects)][prev_month].sum()
        
        # Сумма отгрузки пролонгированных проектов в текущем месяце
        prolong_sum_k1 = manager_financial_grouped[manager_financial_grouped['id'].isin(ended_projects)][current_month].sum()
        
        # Коэффициент пролонгации в первый месяц
        
        if last_month_sum_k1 > 0:
            prolongation_coefficient_k1 = prolong_sum_k1 / last_month_sum_k1
        else:
            prolongation_coefficient_k1 = 0  

# ---------------------------------------------------------------------------------
        # Подсчитаем второй коэффицент пролонгаций по сотрудникам
        
        # Находим проекты, завершившиеся два месяца назад
        if month_idx >= 2:
            
            two_month_ago = columns_month[month_idx - 2]
            
            ended_projects_two_months_ago = manager_prolong[manager_prolong['month'] == two_month_ago]['id'].unique()
    
            # Находим проекты, которые на следующий месяц не имели отгрузку
            not_prolonged_first_month = manager_financial_grouped[manager_financial_grouped['id'].isin(ended_projects_two_months_ago) & \
                                    (manager_financial_grouped[prev_month] == 0)]['id'].unique()
        
            # Сумма отгрузки в месяц завершения
            two_month_ago_sum_k2 = manager_financial_grouped[manager_financial_grouped['id'].isin(not_prolonged_first_month )][two_month_ago].sum()
        
            # Сумма отгрузки в текущйи месяц
            prolonged_sum_two_k2 = manager_financial_grouped[manager_financial_grouped['id'].isin(not_prolonged_first_month)][current_month].sum()
            
            if two_month_ago_sum_k2 > 0:
                prolongation_coefficient_k2 = prolonged_sum_two_k2 / two_month_ago_sum_k2
            else:
                prolongation_coefficient_k1 = 0  
    
            prolongation_year_manager.append({
                'Менеджер': manager, 
                'Месяц': current_month,
                'Коэффициент пролонгаций 1': prolongation_coefficient_k1,
                'Коэффициент пролонгаций 2': prolongation_coefficient_k2})

        
prolongation_manager_year_df = pd.DataFrame(prolongation_year_manager)
# Годовые коэффициенты по менеджерам
annual_manager_ratios = prolongation_manager_year_df.groupby('Менеджер')[['Коэффициент пролонгаций 1', 'Коэффициент пролонгаций 2']].mean().reset_index()


In [696]:
annual_first_ratio = prolongation_year_df['Коэффициент пролонгаций 1'].mean()
annual_second_ratio = prolongation_year_df['Коэффициент пролонгаций 2'].mean()

print(f'Первый коэффициент пролонгаций для всей компании в 2023 году равен {round(annual_first_ratio,2)}')
print(f'Второй коэффициент пролонгаций для всей компании в 2023 году равен {round(annual_second_ratio,2)}')

Первый коэффициент пролонгаций для всей компании в 2023 году равен 0.54
Второй коэффициент пролонгаций для всей компании в 2023 году равен 0.08


In [698]:
# Сохраняем все данные в csv для дальнейшей работы в Excel

prolongation_year_df.to_csv('prolong_for_all_company.csv')
prolongation_manager_year_df.to_csv('prolong_for_all_manager_month.csv')
annual_manager_ratios.to_csv('prolong_for_all_manager_year.csv')
